# Aprendizado de Máquina — Lista prática 08

## Métricas para Classificação

**Gabriel Sanfins** &nbsp;·&nbsp; gabrielsanfins@id.uff.br

---

Marta trabalha na área de relacionamento de um banco. O banco quer prever quais
clientes farão uma certa transação, que ele não revela, e a equipe de Marta vai entrar
em contato com os clientes que um modelo apontar. Os dados são os do
`bank_train_redux.csv`, os mesmos do laboratório: 200 covariáveis anônimas e a
resposta $Y$, que vale 1 para quem fez a transação, cerca de 10% dos clientes. A lista
usa outra divisão entre treino e teste, e por isso os números diferem um pouco dos do
laboratório.

Ao longo da lista, $\widehat p(\mathbf{x})$ é a probabilidade de $Y=1$ estimada pela
regressão logística, e o cliente é declarado positivo quando
$\widehat p(\mathbf{x}) \ge t$, para um corte $t$ escolhido. Os exercícios seguem a
ordem do laboratório: a acurácia, a matriz de confusão, a escolha do corte e a curva
ROC.

Cada lacuna está marcada com `...`. Substitua cada uma pela sua resposta e rode a
célula.

---
## 1. Importando os pacotes

In [ ]:
import os
import numpy as np
import pandas as pd
from matplotlib.pyplot import subplots

import sklearn.linear_model as skl
import sklearn.model_selection as skm
from sklearn.metrics import confusion_matrix, roc_auc_score, roc_curve
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

import warnings
warnings.filterwarnings("ignore")

---
## Exercício 1 — a acurácia e o classificador trivial

Quem ajustou o modelo informou a Marta que ele acerta 91,55% dos clientes de teste.
Antes de repassar o número, ela quer saber quanto dele se deve ao modelo, e para isso
o compara com o classificador trivial, que declara todo cliente negativo sem olhar
covariável nenhuma.

A célula a seguir lê os dados, com a mesma limpeza do laboratório, e separa 30% dos
clientes para teste. (a) Preencha o argumento que faz a divisão preservar, no treino e
no teste, a proporção de positivos.

In [ ]:
_nome = "bank_train_redux.csv"

# procura em dois lugares, sem baixar nada da internet: a pasta deste
# notebook primeiro ou então ../../recursos/dados/
_lugares = [_nome, os.path.join("..", "..", "recursos", "dados", _nome)]
_caminho = next((c for c in _lugares if os.path.exists(c)), None)

if _caminho is None:
    raise FileNotFoundError(
        f"nao encontrei '{_nome}'. Procurei nesta pasta e em "
        "../../recursos/dados/. Ponha o .csv ao lado deste notebook, "
        "ou mude o caminho se for necessário."
    )

banco = pd.read_csv(_caminho, nrows=40_000)
banco = banco.replace(to_replace=";", value="", regex=True)
banco = banco.rename(columns={"var_199;;;;;;;": "var_199"})
banco["var_199"] = pd.to_numeric(banco["var_199"])

Xb = banco.drop(columns=["ID_code", "target"]).astype(float).values
yb = banco["target"].values

X_tr, X_te, y_tr, y_te = skm.train_test_split(
    Xb, yb, test_size=0.3, random_state=2026, stratify=...)      # (a)

print(f"n = {Xb.shape[0]}, d = {Xb.shape[1]}, prevalencia = {yb.mean():.4f}")
print(f"treino: {len(y_tr)} clientes, {int(y_tr.sum())} positivos")
print(f"teste : {len(y_te)} clientes, {int(y_te.sum())} positivos")

O modelo é o do laboratório: a regressão logística, com as covariáveis padronizadas
pelas médias e pelos desvios-padrão do treino, as duas etapas encadeadas num
`Pipeline`. O método `predict_proba` devolve uma coluna por classe, na ordem de
`modelo.classes_`, que aqui é `[0, 1]`. (b) Extraia $\widehat p$, a probabilidade
estimada de $Y=1$. (c) Calcule a acurácia do classificador trivial.

In [ ]:
modelo = Pipeline([("sc", StandardScaler()),
                   ("lg", skl.LogisticRegression(max_iter=2000))]).fit(X_tr, y_tr)
p = modelo.predict_proba(X_te)[:, ...]                            # (b)
pred = (p >= 0.5).astype(int)

acuracia = np.mean(pred == y_te)
trivial = ...                                    # (c)
print(f"acuracia do modelo, corte 0,5   : {acuracia:.4f}")
print(f"acuracia de 'ninguem e positivo': {trivial:.4f}")
print(f"diferenca                       : {acuracia - trivial:+.4f}")
print(f"\nclientes declarados positivos: {pred.sum()} de {len(y_te)}")

---
## Exercício 2 — a matriz de confusão e as taxas

Para ver o que o modelo faz em cada classe, Marta separa os acertos e os erros do
corte 0,5 em quatro contagens: os verdadeiros positivos (VP), os falsos positivos
(FP), os falsos negativos (FN) e os verdadeiros negativos (VN). A função
`confusion_matrix` devolve essas contagens com a classe real nas linhas e a prevista
nas colunas, e o `ravel` as lê linha a linha, na ordem VN, FP, FN, VP. Na aula, a
orientação é a transposta: a classe prevista, $\widehat Y$, nas linhas, e a real, $Y$,
nas colunas. (a) Monte a tabela na orientação da aula.

In [ ]:
mc = confusion_matrix(y_te, pred)          # o scikit-learn devolve o real nas linhas
vn, fp, fn, vp = mc.ravel()
P, N = vp + fn, fp + vn
n = len(y_te)

tabela = pd.DataFrame(..., index=["previsto 0", "previsto 1"],     # (a)
                      columns=["real 0", "real 1"])
print(tabela)
print(f"\nP = {P} positivos e N = {N} negativos, n = {n}")

Cada taxa divide uma contagem pelo total de uma coluna ou de uma linha. Como o teste
não foi usado no ajuste, cada uma estima uma probabilidade sobre um cliente novo. A
sensibilidade, $VP/P$, estima $\mathbb{P}(\widehat Y=1\mid Y=1)$; a taxa de falsos
positivos, $FP/N$, estima $\mathbb{P}(\widehat Y=1\mid Y=0)$, e a especificidade,
$VN/N$, é o seu complemento; a precisão, $VP/(VP+FP)$, estima
$\mathbb{P}(Y=1\mid \widehat Y=1)$, condicionando na decisão em vez de na classe; e a
taxa de erro, $(FP+FN)/n$, estima $\mathbb{P}(\widehat Y\neq Y)$. (b) Complete a
sensibilidade, a taxa de falsos positivos e a precisão.

In [ ]:
sens = ...                                                 # (b)
taxa_fp = ...                                              # (b)
espec = vn / N
precisao = ...                                     # (b)
taxa_erro = (fp + fn) / n

print(f"sensibilidade            : {sens:.4f}")
print(f"taxa de falsos positivos : {taxa_fp:.4f}")
print(f"especificidade           : {espec:.4f}")
print(f"precisao                 : {precisao:.4f}")
print(f"taxa de erro             : {taxa_erro:.4f}")

A acurácia mistura essas taxas, e a conta mostra em que proporção. Como $n = P + N$,

$$\frac{VP+VN}{n} = \frac{N}{n}\cdot\frac{VN}{N} + \frac{P}{n}\cdot\frac{VP}{P}:$$

a acurácia é a média da especificidade e da sensibilidade, ponderada pelas proporções
das duas classes no teste. A mesma conta mede o ganho sobre o classificador trivial,
cuja acurácia é $N/n$. Como $VN = N - FP$,

$$\frac{VP+VN}{n} - \frac{N}{n} = \frac{VP-FP}{n}.$$

Os dois classificadores só discordam nos clientes que o modelo declara positivos: em
cada verdadeiro positivo o modelo acerta e o trivial erra, e em cada falso positivo
acontece o contrário. (c) Confira as duas identidades.

In [ ]:
print(f"acuracia                       : {acuracia:.4f}")
print(f"media ponderada das duas taxas : {...:.4f}")      # (c)
print(f"   parcela dos negativos       : {N / n * espec:.4f}")
print(f"   parcela dos positivos       : {P / n * sens:.4f}")
print(f"\nganho sobre o trivial          : {acuracia - trivial:.4f}")
print(f"(VP - FP)/n                    : {...:.4f}")                     # (c)

---
## Exercício 3 — o corte é uma escolha

A equipe de Marta consegue fazer 1 000 contatos por mês, e o corte 0,5 aponta só 484
clientes. Para usar a capacidade inteira, Marta ordena os clientes por $\widehat p$ e
fica com os 1 000 primeiros. Isso é escolher outro corte: $t$ igual ao milésimo maior
valor de $\widehat p$, de modo que exatamente 1 000 clientes tenham $\widehat p \ge t$,
já que os 12 000 escores do teste são todos distintos. Para avaliar a regra, Marta a
aplica aos clientes de teste, como se fossem os clientes de um mês.

A função `taxas` refaz as contas do Exercício 2 para um corte qualquer. (a) Complete a
taxa de erro na função. (b) Calcule o corte que seleciona os 1 000 clientes de maior
$\widehat p$.

In [ ]:
def taxas(y, prob, t):
    "Contagens e taxas da regra que declara positivo quando prob >= t."
    vn_t, fp_t, fn_t, vp_t = confusion_matrix(y, (prob >= t).astype(int)).ravel()
    return {"declarados": vp_t + fp_t,
            "sensibilidade": vp_t / (vp_t + fn_t),
            "taxa de falsos positivos": fp_t / (fp_t + vn_t),
            "precisao": vp_t / (vp_t + fp_t),
            "taxa de erro": ...}                  # (a)


t_mil = ...                                     # (b)

for nome, t in [("corte 0,5", 0.5), ("1 000 contatos", t_mil)]:
    r = taxas(y_te, p, t)
    print(f"{nome:15s} t = {t:.4f}  declarados {r['declarados']:5d}  "
          f"sensibilidade {r['sensibilidade']:.4f}  FPR {r['taxa de falsos positivos']:.4f}  "
          f"precisao {r['precisao']:.4f}  erro {r['taxa de erro']:.4f}")

Para ver a troca inteira, a célula seguinte varre os cortes de 0,02 a 0,95 e desenha,
para cada um, a sensibilidade, a precisão e a taxa de erro, com os dois cortes acima
marcados. Ela localiza também o corte de menor taxa de erro na grade e conta os
clientes cujo $\widehat p$ fica entre 0,5 e esse corte.

In [ ]:
cortes = np.linspace(0.02, 0.95, 200)
curvas = pd.DataFrame([taxas(y_te, p, t) for t in cortes], index=cortes)
t_erro = curvas["taxa de erro"].idxmin()
entre = (p >= 0.5) & (p < t_erro)
print(f"menor taxa de erro da grade: {curvas['taxa de erro'].min():.4f}, no corte {t_erro:.3f}")
print(f"clientes com p entre 0.500 e {t_erro:.3f}: {entre.sum()}, "
      f"dos quais {y_te[entre].sum()} positivos")

fig, ax = subplots(figsize=(6.8, 3.4))
for nome in ["sensibilidade", "precisao", "taxa de erro"]:
    ax.plot(cortes, curvas[nome], label=nome)
ax.axvline(0.5, ls=":", color="gray", label="corte 0,5")
ax.axvline(t_mil, ls="--", color="gray", label="1 000 contatos")
ax.set_xlabel("corte $t$")
ax.set_ylabel("valor")
ax.legend(fontsize=7.5, loc="center left", bbox_to_anchor=(1.01, 0.5), frameon=False)
fig.tight_layout()

---
## Exercício 4 — a curva ROC e a AUC

A capacidade da equipe muda de um mês para outro, e com ela o corte. Para avaliar o
modelo sem fixar um corte, Marta usa a curva ROC, o conjunto dos pontos (taxa de falsos
positivos, sensibilidade) obtidos quando $t$ percorre $[0,1]$, e a AUC, a área sob ela.
(a) Calcule a AUC. A célula marca na curva os dois cortes do Exercício 3.

In [ ]:
fpr, tpr, _ = roc_curve(y_te, p)
auc = ...                                  # (a)
print(f"AUC = {auc:.4f}")

fig, ax = subplots(figsize=(5, 4))
ax.plot(fpr, tpr, lw=1.6, label="logistica")
ax.plot([0, 1], [0, 1], ls="--", lw=1, color="gray", label="sorteio")
for nome, t, marca in [("corte 0,5", 0.5, "o"), ("1 000 contatos", t_mil, "s")]:
    r = taxas(y_te, p, t)
    ax.plot(r["taxa de falsos positivos"], r["sensibilidade"], marca, color="crimson",
            ms=6, label=nome)
ax.set_xlabel("taxa de falsos positivos")
ax.set_ylabel("sensibilidade")
ax.legend(fontsize=8, loc="lower right")
fig.tight_layout()

O laboratório conferiu por sorteio que a AUC é uma probabilidade: se $Z_1$ e $Z_0$ são
os escores de um positivo e de um negativo sorteados de forma independente,

$$\mathrm{AUC} = \mathbb{P}(Z_1 > Z_0) + \tfrac12\,\mathbb{P}(Z_1 = Z_0).$$

No teste há $P \cdot N = 1\,182 \times 10\,818$ pares, cerca de 12,8 milhões, e a conta
pode ser feita sem sorteio: a fração dos pares em que o positivo tem escore maior, com
os empates contando meio. (b) Complete a comparação de todos os pares.

In [ ]:
pos, neg = p[y_te == 1], p[y_te == 0]
maior = ...                           # (b)
empate = pos[:, None] == neg[None, :]

print(f"pares comparados                         : {maior.size}")
print(f"pares empatados                          : {empate.sum()}")
print(f"fracao com Z1 > Z0, empates contando meio: {maior.mean() + 0.5 * empate.mean():.6f}")
print(f"roc_auc_score                            : {auc:.6f}")

Por fim, uma propriedade que importa a Marta porque a prevalência muda: num mês de
campanha, a fração de clientes que faz a transação pode ser outra. A sensibilidade e a
taxa de falsos positivos condicionam na classe verdadeira e, por isso, não dependem da
prevalência: se só a proporção de positivos mudar, mantida a distribuição das
covariáveis dentro de cada classe, o que elas estimam fica igual, e a curva ROC também.
A precisão condiciona na decisão e depende dela. Pelo teorema de Bayes, como nas notas,
com $\pi_1 = \mathbb{P}(Y=1)$,

$$\mathbb{P}(Y=1\mid \widehat Y=1)
  = \frac{\mathrm{TPR}\,\pi_1}{\mathrm{TPR}\,\pi_1 + \mathrm{FPR}\,(1-\pi_1)},$$

em que $\mathrm{TPR}$ é a sensibilidade e $\mathrm{FPR}$, a taxa de falsos positivos.

Para ver isso nos dados, a célula a seguir forma um conjunto com prevalência de 50%: os
1 182 positivos do teste e 1 182 negativos sorteados, sem reposição, entre os 10 818.
Sortear os negativos ao acaso muda a proporção das classes sem mudar a distribuição das
covariáveis dentro de cada uma. (c) Sorteie, com `rng.choice`, os índices dos
negativos. (d) Complete a precisão pela fórmula acima.

In [ ]:
rng = np.random.default_rng(2026)
i_pos = np.flatnonzero(y_te == 1)
i_neg = ...    # (c)
i_eq = np.concatenate([i_pos, i_neg])

for nome, yy, pp in [("teste inteiro", y_te, p), ("prevalencia 50%", y_te[i_eq], p[i_eq])]:
    r = taxas(yy, pp, 0.5)
    pi1, s, f = yy.mean(), r["sensibilidade"], r["taxa de falsos positivos"]
    formula = ...                                # (d)
    print(f"{nome:16s} prevalencia {pi1:.4f}  AUC {roc_auc_score(yy, pp):.4f}  "
          f"sensibilidade {s:.4f}  FPR {f:.4f}  precisao {r['precisao']:.4f}  "
          f"(pela formula {formula:.4f})")

> **Sua vez.** A AUC do conjunto equilibrado depende de quais negativos foram
> sorteados. Repita o sorteio com 100 sementes diferentes, guarde a AUC de cada
> conjunto e veja quanto ela varia. A diferença entre 0,8488 e 0,8529 é grande diante
> dessa variação?

---
## O que ficou

| Exercício | O que você mediu |
|---|---|
| 1 | acurácia 0,9155, contra 0,9015 do classificador trivial; no corte 0,5 o modelo declara positivos 484 clientes, para 1 182 positivos |
| 2 | sensibilidade 0,276, taxa de falsos positivos 0,015 e precisão 0,674; o ganho de acurácia sobre o trivial é $(VP-FP)/n$ |
| 3 | com 1 000 contatos (corte 0,3234), a sensibilidade sobe para 0,435 e a precisão cai para 0,514; a taxa de erro é mínima perto de 0,5, onde a curva é quase plana |
| 4 | AUC 0,8529, exatamente a fração dos 12,8 milhões de pares em que o positivo tem escore maior; com prevalência de 50%, a sensibilidade não muda e a precisão vai a 0,956 |

**A seguir.** A Aula 09 traz um classificador construído sobre uma ideia geométrica, a
margem, e que não estima probabilidades.